# ML-KEM-768 KAT Benchmark Notebook

Combines the **correctness** guarantee of `ml_kem_kat_test.ipynb` with the **per-op timing** of `ml_kem_bench.ipynb`. For each iteration, inputs come from `KAT_768.txt` (deterministic, NIST-blessed) and each produced output is validated against the expected value on the fly — any mismatch aborts the run.

Reported per op (same schema as `ml_kem_bench.ipynb`):

- **HW cycles** (median / min / max) — on-chip counter.
- **HW latency** in µs @ 100 MHz.
- **Wall time** seen by Python — includes register writes, cache ops, polling.
- **PYNQ overhead** — wall − hw, software control-path cost.
- **Throughput** in ops/s, single-thread pulse-then-poll.

## When to use this vs the random bench

| Notebook | Inputs | Purpose |
|---|---|---|
| `ml_kem_bench.ipynb` | `secrets.token_bytes(32)` | fast performance scan, wide input variance |
| **`ml_kem_kat_bench.ipynb`** (this) | NIST KAT vectors | perf + correctness in one pass, reproducible |
| `ml_kem_kat_test.ipynb` | NIST KAT vectors | conformance only, no timing stats |

## Op selection

| `op` value | Behavior |
|---|---|
| `"all"` | Run KeyGen, Encaps, Decaps, Full KEM |
| `"keygen"` | KeyGen only — input: `(d, z)`, check: `pk, sk` |
| `"encaps"` | Encaps only — input: `(pk, m)` from KAT, check: `ct, ss` |
| `"decaps"` | Decaps only — input: `(sk, ct)` from KAT, check: `ss` |
| `"full"` | KeyGen + Encaps + Decaps per iter, ss round-trip asserted |

If `n_iters > len(vectors)`, the notebook cycles through the KAT file with wrap-around (`vectors[i % len(vectors)]`). Each iteration still reports fresh HW cycles, so you get stable median stats without needing 1000 unique KAT vectors.

In [ ]:
import os
import statistics
import time

from ml_kem_driver import MLKem768, cycles_to_us


def parse_kat_file(path):
    """Parse NIST KAT file into a list of dicts (same format as ml_kem_kat_test)."""
    vectors = []
    current = {}
    with open(path, "r") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                if current:
                    vectors.append(current)
                    current = {}
                continue
            if "=" not in line:
                continue
            key, _, val = line.partition("=")
            key = key.strip()
            val = val.strip()
            try:
                current[key] = bytes.fromhex(val)
            except ValueError:
                current[key] = val
    if current:
        vectors.append(current)
    return vectors

In [ ]:
def bench_op(label, fn, n):
    """Run fn(iter_idx) n times. fn must return (cycles, ok, err_msg).

    If ok=False on any iter, we print the error and still keep the cycle
    sample so the run continues (use stop_on_fail=True in wrappers to abort).
    """
    cycles_list = []
    wall_list = []
    fail_ct = 0
    first_fail = None

    for i in range(n):
        t0 = time.monotonic()
        cyc, ok, err = fn(i)
        wall_list.append(time.monotonic() - t0)
        cycles_list.append(cyc)
        if not ok:
            fail_ct += 1
            if first_fail is None:
                first_fail = (i, err)

    c_med = statistics.median(cycles_list)
    c_min = min(cycles_list)
    c_max = max(cycles_list)
    w_med = statistics.median(wall_list)
    w_min = min(wall_list)
    w_max = max(wall_list)
    throughput = n / sum(wall_list) if sum(wall_list) > 0 else 0.0

    print(f"  {label}")
    print(f"    HW cycles   : median={int(c_med):6d}  min={c_min:6d}  max={c_max:6d}")
    print(f"    HW latency  : median={cycles_to_us(c_med):6.1f} us  (= {int(c_med)} cyc @ 100 MHz)")
    print(f"    Wall time   : median={w_med*1e6:6.1f} us  min={w_min*1e6:6.1f} us  max={w_max*1e6:6.1f} us")
    print(f"    PYNQ ovhd   : ~{(w_med*1e6 - cycles_to_us(c_med)):6.1f} us  (wall - hw)")
    print(f"    Throughput  : {throughput:6.1f} ops/s  (pulse-then-poll, single-thread)")
    print(f"    KAT correct : {n - fail_ct}/{n} pass")
    if fail_ct:
        idx, msg = first_fail
        print(f"    FIRST FAIL  : vec #{idx} -- {msg}")

    return {
        "label": label,
        "cycles": cycles_list,
        "wall_s": wall_list,
        "cycle_median": c_med,
        "wall_median_s": w_med,
        "throughput_ops_s": throughput,
        "fail_count": fail_ct,
        "first_fail": first_fail,
    }

In [ ]:
# Per-op bench wrappers driven by KAT vectors (wrap around if n > len(vectors)).

def bench_keygen_kat(kem, vectors, n):
    print(f"--- KeyGen (KAT-driven, {len(vectors)} unique vectors) ---")

    def step(i):
        v = vectors[i % len(vectors)]
        pk, sk, c = kem.keygen(v["d"], v["z"])
        if pk != v["pk"]:
            return c, False, f"pk mismatch at vec #{i % len(vectors)}"
        if sk != v["sk"]:
            return c, False, f"sk mismatch at vec #{i % len(vectors)}"
        return c, True, None

    return bench_op("KeyGen", step, n)


def bench_encaps_kat(kem, vectors, n):
    print(f"--- Encaps (KAT-driven, {len(vectors)} unique vectors) ---")

    def step(i):
        v = vectors[i % len(vectors)]
        ct, ss, c = kem.encaps(v["pk"], v["m"])
        if ct != v["ct"]:
            return c, False, f"ct mismatch at vec #{i % len(vectors)}"
        if ss != v["ss"]:
            return c, False, f"ss mismatch at vec #{i % len(vectors)}"
        return c, True, None

    return bench_op("Encaps", step, n)


def bench_decaps_kat(kem, vectors, n):
    print(f"--- Decaps (KAT-driven match branch, {len(vectors)} unique vectors) ---")

    def step(i):
        v = vectors[i % len(vectors)]
        ss, c = kem.decaps(v["sk"], v["ct"])
        if ss != v["ss"]:
            return c, False, f"ss mismatch at vec #{i % len(vectors)}"
        return c, True, None

    return bench_op("Decaps", step, n)


def bench_full_kat(kem, vectors, n):
    print(f"--- Full KEM (KAT-driven KG+Enc+Dec per iter, {len(vectors)} unique vectors) ---")

    def step(i):
        v = vectors[i % len(vectors)]
        pk, sk, c1 = kem.keygen(v["d"], v["z"])
        if pk != v["pk"] or sk != v["sk"]:
            return c1, False, f"KG mismatch at vec #{i % len(vectors)}"
        ct, ss_enc, c2 = kem.encaps(pk, v["m"])
        if ct != v["ct"] or ss_enc != v["ss"]:
            return c1 + c2, False, f"Encaps mismatch at vec #{i % len(vectors)}"
        ss_dec, c3 = kem.decaps(sk, ct)
        if ss_dec != v["ss"]:
            return c1 + c2 + c3, False, f"Decaps mismatch at vec #{i % len(vectors)}"
        return c1 + c2 + c3, True, None

    return bench_op("Full KEM", step, n)


OP_DISPATCH = {
    "keygen": bench_keygen_kat,
    "encaps": bench_encaps_kat,
    "decaps": bench_decaps_kat,
    "full":   bench_full_kat,
}

## Config — edit these and re-run cells below

In [ ]:
script_dir = os.getcwd()
BITSTREAM_DIR = "/root/jupyter_notebooks/verilog_ML_KEM/bitstream"

bitfile  = os.environ.get("ML_KEM_BIT", os.path.join(BITSTREAM_DIR, "ml_kem_bd.bit"))
kat_path = os.environ.get("KAT_FILE",   os.path.join(script_dir, "KAT_768.txt"))

n_iters   = 100           # total iterations per op; wraps around KAT pool if > len(vectors)
n_vectors = None          # None => use full KAT file; else take first K vectors
op        = "all"         # one of: "all", "keygen", "encaps", "decaps", "full"

assert op in ("all", "keygen", "encaps", "decaps", "full"), f"unknown op: {op!r}"
assert os.path.exists(bitfile), f"bitfile not found on board: {bitfile}"
assert os.path.exists(bitfile.replace('.bit', '.hwh')), "missing .hwh next to .bit"
assert os.path.exists(kat_path), f"KAT file not found: {kat_path}"

print(f"bitfile   : {bitfile}")
print(f"kat_path  : {kat_path}")
print(f"n_iters   : {n_iters}")
print(f"n_vectors : {n_vectors}")
print(f"op        : {op}")

In [ ]:
# Load KAT pool once (shared across ops)
vectors = parse_kat_file(kat_path)
if n_vectors is not None:
    vectors = vectors[:n_vectors]
print(f"Loaded {len(vectors)} KAT vector(s). Bench will run {n_iters} iter(s) per op.")
if n_iters > len(vectors):
    print(f"  (iters > vectors => will wrap around {n_iters // len(vectors)}x + {n_iters % len(vectors)} partial)")

In [ ]:
# Open driver (run once per session). Call kem.close() in the cleanup cell
# before re-running this cell.
kem = MLKem768(bitfile)
print("Driver loaded.")

In [ ]:
# Run selected op(s). Results preserved in `results` dict for post-hoc analysis.
results = {}
if op == "all":
    for name in ["keygen", "encaps", "decaps", "full"]:
        results[name] = OP_DISPATCH[name](kem, vectors, n_iters)
        print()
else:
    results[op] = OP_DISPATCH[op](kem, vectors, n_iters)

# Aggregate correctness — fail the cell loudly if any KAT mismatch occurred
total_fail = sum(r["fail_count"] for r in results.values())
if total_fail:
    raise AssertionError(
        f"KAT regression FAILED: {total_fail} mismatch(es) across op(s). See per-op output above."
    )
print("All KAT checks PASSED.")

## Optional: inspect raw distribution

`results[op_name]["cycles"]` and `results[op_name]["wall_s"]` hold per-iteration samples. Useful for histogram plotting, percentile checks, or empirical constant-time verification (`max(cycles) - min(cycles)` should be 0 for decaps over a KAT pool).

In [ ]:
# Empirical constant-time gauge: cycle spread per op across the iter pool
print("Op        cyc_min   cyc_max   spread   (spread should be 0 for constant-time ops)")
for name, r in results.items():
    cmin = min(r["cycles"])
    cmax = max(r["cycles"])
    print(f"  {name:8s}  {cmin:6d}    {cmax:6d}    {cmax - cmin:5d}")

In [ ]:
# CSV-friendly summary row: op, cyc_median, hw_us, wall_us, ops_per_s, kat_pass/total
print("op,cyc_median,hw_us,wall_us,throughput_ops_s,kat_pass,kat_total")
for name, r in results.items():
    total = len(r["cycles"])
    print(f"{name},"
          f"{int(r['cycle_median'])},"
          f"{cycles_to_us(r['cycle_median']):.2f},"
          f"{r['wall_median_s']*1e6:.2f},"
          f"{r['throughput_ops_s']:.2f},"
          f"{total - r['fail_count']},"
          f"{total}")

In [ ]:
# Cleanup: free CMA buffers. Skip if you intend to keep using `kem`.
kem.close()
print("Driver closed.")

## How to interpret results

- **`KAT correct`** — must read `n/n` for every op. Any mismatch means a regression; the final cell raises `AssertionError`.
- **`HW cycles` median vs random-input bench** — should match `ml_kem_bench.ipynb` almost exactly (same RTL, same param set). If the median differs by more than a few cycles, something is wrong.
- **`cycle spread` (max − min)** — on a constant-time core this should be **0** for KeyGen/Encaps/Decaps (FIPS 203 has no secret-dependent branches when implemented correctly). Non-zero spread = either non-constant-time bug or a measurement artifact from concurrent PS activity; investigate.
- **`Wall time` / `PYNQ ovhd`** — same interpretation as `ml_kem_bench.ipynb`; software control-path cost, independent of input choice.
- **Throughput** — ops/s with KAT replay will be marginally lower than random bench because of the per-iter expected-value comparison in Python (a few extra µs per op). The HW numbers are unaffected.